# Generacion con LLM: Gobernante vs Especialista, 2 estilos de prompt

Usa los contextos **ya recuperados** por las dos configuraciones de retrieve elegidas (OpenAI `text-embedding-3-large` denso cs800, y `BAAI/bge-m3` hibrido cs800) — no se vuelve a hacer retrieval aqui. Se filtra primero a las preguntas donde AMBOS pipelines tuvieron exito en RAGAS (`llm_context_precision_with_reference > 0` y `context_recall > 0`), para no pedirle a un LLM que responda con un contexto que ya sabemos que no sirve.

**Modelos** (mismo proveedor a proposito, para aislar el efecto de capacidad/tamano sin mezclar el "estilo de casa" de distintos proveedores):
- **Gobernante** — `anthropic/claude-sonnet-5`: alta capacidad, 1M de contexto, $2/$10 por MTok (se prefirio a Opus 5 por costo, manteniendo el mismo proveedor).
- **Especialista** — `anthropic/claude-haiku-4.5`: eficiente, 200K de contexto, $1/$5 por MTok (2x mas barato que Sonnet 5 en ambas direcciones).

**Estilos de prompt** (2, deliberadamente distintos, cada uno con una hipotesis sobre a cual modelo favorece mas):
- `directo`: instruccion minima, sin scaffolding — hipotesis: favorece al modelo de alta capacidad.
- `estructurado`: pide identificar fragmentos relevantes, evaluar cobertura, y responder en un formato fijo — hipotesis: compensa a un modelo mas chico con una guia explicita.

2 pipelines de retrieve x 2 modelos x 2 prompts = **8 combinaciones**, cada una corrida sobre el mismo conjunto filtrado de preguntas.

**Nota metodologica sobre resiliencia**: a diferencia del notebook de RAGAS (donde el LLM juez podia tener un modelo de respaldo, porque cualquier juez razonable sirve), aqui el modelo **es** la variable que se esta midiendo. Usar el fallback de modelos de OpenRouter contaminaria la comparacion sin que se note (una respuesta "del Especialista" podria en realidad venir de otro modelo). Por eso la resiliencia aqui es solo reintentos con backoff **sobre el mismo modelo**, nunca sustitucion.

**Variables de entorno:** `OPENROUTER_API_KEY`.


In [ ]:
%pip install -q openai pandas

## 1. Filtrar a las preguntas donde ambos pipelines de retrieve tuvieron exito

Se usa `evaluacion_ragas_detalle.csv` (generado por el notebook de evaluacion) para identificar, por pregunta, si cada pipeline tuvo `llm_context_precision_with_reference > 0` y `context_recall > 0`. Solo se generan respuestas para la interseccion.


In [1]:
import json
import pandas as pd
from pathlib import Path

RUTA_EVALUACION_DETALLE = "./validacion/evaluacion_ragas_detalle.csv"
RUTA_RETRIEVAL_OPENAI = "./validacion/resultados_retrieval.json"              # 9 configuraciones; se usa openai cs800
RUTA_RETRIEVAL_BGE = "./validacion/resultados_retrieval_bge_m3_hibrido.json"  # BGE-m3 hibrido cs800, k=5

ETIQUETA_OPENAI = "openai/text-embedding-3-large_cs800"
ETIQUETA_BGE = "BAAI/bge-m3_cs800_hibrido"

evaluacion = pd.read_csv(RUTA_EVALUACION_DETALLE)


def preguntas_con_score_positivo(etiqueta):
    sub = evaluacion[evaluacion["configuracion"] == etiqueta]
    validas = sub[(sub["llm_context_precision_with_reference"] > 0) & (sub["context_recall"] > 0)]
    return set(validas["user_input"])


preguntas_validas_openai = preguntas_con_score_positivo(ETIQUETA_OPENAI)
preguntas_validas_bge = preguntas_con_score_positivo(ETIQUETA_BGE)
preguntas_validas = preguntas_validas_openai & preguntas_validas_bge

print(f"Preguntas con score>0 en OpenAI cs800:      {len(preguntas_validas_openai)}")
print(f"Preguntas con score>0 en BGE-m3 hibrido:     {len(preguntas_validas_bge)}")
print(f"Interseccion (las que se usaran para generar): {len(preguntas_validas)}")

Preguntas con score>0 en OpenAI cs800:      245
Preguntas con score>0 en BGE-m3 hibrido:     219
Interseccion (las que se usaran para generar): 196


In [2]:
def cargar_resultados_config(ruta, etiqueta_config=None):
    with open(ruta, "r", encoding="utf-8") as f:
        contenido = json.load(f)
    if "configuraciones" in contenido:
        for c in contenido["configuraciones"]:
            etiqueta = f"{c['modelo_embedding']}_cs{c['chunk_size']}"
            if etiqueta == etiqueta_config:
                return c["resultados"]
        raise ValueError(f"No se encontro la configuracion \'{etiqueta_config}\' en {ruta}")
    return contenido["resultados"]


resultados_openai_todos = cargar_resultados_config(RUTA_RETRIEVAL_OPENAI, ETIQUETA_OPENAI)
resultados_bge_todos = cargar_resultados_config(RUTA_RETRIEVAL_BGE)

resultados_openai = sorted(
    (r for r in resultados_openai_todos if r["pregunta"] in preguntas_validas),
    key=lambda r: r["pregunta"],
)
resultados_bge = sorted(
    (r for r in resultados_bge_todos if r["pregunta"] in preguntas_validas),
    key=lambda r: r["pregunta"],
)

# Mismo orden y mismas preguntas en ambos, para que la comparacion entre pipelines sea 1 a 1
assert [r["pregunta"] for r in resultados_openai] == [r["pregunta"] for r in resultados_bge]
print(f"{len(resultados_openai)} preguntas listas, con 2 conjuntos de contexto cada una (OpenAI y BGE hibrido).")

PIPELINES_RETRIEVE = {
    "openai_cs800": resultados_openai,
    "bge_m3_hibrido_cs800": resultados_bge,
}

196 preguntas listas, con 2 conjuntos de contexto cada una (OpenAI y BGE hibrido).


## 2. Modelos LLM y estilos de prompt


In [4]:
MODELOS_LLM = {
    # "Gobernante": alta capacidad de razonamiento, ventana de contexto enorme.
    # Se uso Sonnet 5 en vez de Opus 5 (mismo proveedor, mismo rol) por costo:
    # $2/$10 por MTok vs $5/$25 de Opus 5 (2.5x mas barato en ambas direcciones),
    # manteniendo la comparacion dentro de la misma familia de modelos.
    "claude-sonnet-5": {
        "openrouter_id": "anthropic/claude-sonnet-5",
        "rol": "gobernante",
    },
    # "Especialista": eficiente, barato, ventana de contexto de sobra para RAG.
    "claude-haiku-4.5": {
        "openrouter_id": "anthropic/claude-haiku-4.5",
        "rol": "especialista",
    },
}


def formatear_contexto(contextos):
    return "\n\n".join(f"--- Fragmento {i + 1} ---\n{c}" for i, c in enumerate(contextos))


def prompt_directo(pregunta, contextos):
    return f"""Responde la siguiente pregunta usando UNICAMENTE la informacion del contexto. Si el contexto no contiene la respuesta, dilo explicitamente en vez de inventar o completar con conocimiento externo.

Contexto:
{formatear_contexto(contextos)}

Pregunta: {pregunta}

Respuesta:"""


def prompt_estructurado(pregunta, contextos):
    return f"""Eres un asistente que responde preguntas basandote ESTRICTAMENTE en el contexto proporcionado, sin usar conocimiento externo. Sigue estos pasos antes de responder:

1. Identifica que fragmentos del contexto son relevantes para la pregunta.
2. Verifica si, con esos fragmentos, la pregunta se puede responder de forma completa, parcial, o no se puede responder.
3. Redacta la respuesta final basandote unicamente en lo que identificaste en el paso 1.

Contexto:
{formatear_contexto(contextos)}

Pregunta: {pregunta}

Responde siguiendo EXACTAMENTE este formato:
Fragmentos relevantes: <cuales y por que, brevemente>
Cobertura: <completa / parcial / nula>
Respuesta final: <tu respuesta aqui; si la cobertura es nula, escribe "No se encuentra en el contexto">"""


ESTILOS_PROMPT = {
    "directo": prompt_directo,
    "estructurado": prompt_estructurado,
}


import re


def extraer_respuesta_final(texto_generado, nombre_prompt):
    """Para el prompt estructurado, separa la 'Respuesta final:' del resto
    (util para evaluar despues sin el ruido de los pasos intermedios)."""
    if texto_generado is None:
        return None
    if nombre_prompt != "estructurado":
        return texto_generado.strip()
    m = re.search(r"Respuesta final:\s*(.*)", texto_generado, re.DOTALL)
    return m.group(1).strip() if m else texto_generado.strip()

## 3. Cliente OpenRouter con reintentos (sin fallback de modelo)


In [5]:
import os
import time
from openai import OpenAI

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise RuntimeError("Define la variable de entorno OPENROUTER_API_KEY")

_cliente_openrouter = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)


def llamar_llm(openrouter_id, prompt, intentos=5, espera_inicial=5, temperature=0):
    """Reintentos con backoff exponencial SOBRE EL MISMO MODELO. No hay
    fallback a otro modelo a proposito: aqui el modelo es la variable que se
    esta midiendo, no algo sustituible sin contaminar la comparacion."""
    ultimo_error = None
    for intento in range(1, intentos + 1):
        try:
            resp = _cliente_openrouter.chat.completions.create(
                model=openrouter_id,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
            )
            return resp.choices[0].message.content
        except Exception as e:
            ultimo_error = e
            espera = espera_inicial * (2 ** (intento - 1))
            print(f"    intento {intento}/{intentos} fallo ({e}); esperando {espera}s...")
            time.sleep(espera)
    print(f"    se agotaron los reintentos para {openrouter_id}: {ultimo_error}")
    return None

## 4. Ejemplo rapido (sanity check antes de la corrida completa)


In [6]:
pregunta_ejemplo = resultados_openai[0]
prompt_ejemplo = prompt_directo(pregunta_ejemplo["pregunta"], pregunta_ejemplo["contextos_recuperados"])

for nombre_modelo, info in MODELOS_LLM.items():
    respuesta = llamar_llm(info["openrouter_id"], prompt_ejemplo)
    print(f"=== {nombre_modelo} ({info['openrouter_id']}) ===")
    print(f"Pregunta: {pregunta_ejemplo['pregunta']}")
    print(f"Respuesta: {respuesta}")
    print("---")

=== claude-sonnet-5 (anthropic/claude-sonnet-5) ===
Pregunta: Compare la duración en semestres y el total de créditos de los programas de Ingeniería de Sistemas de la Universidad de Antioquia y la Universidad Nacional de Colombia según la Tabla 1.3.
Respuesta: Según la información proporcionada (en los Fragmentos 1 y 2, aunque no se menciona explícitamente como "Tabla 1.3"):

**Universidad de Antioquia** (Medellín) - Programa: Ingeniería de sistemas
- Duración: 10 semestres
- Total de créditos: 180

**Universidad Nacional de Colombia** (Bogotá) - Programa: Ingeniería de sistemas y computación
- Duración: 10 semestres
- Total de créditos: 165

**Comparación:**
Ambos programas tienen la misma duración de **10 semestres**. Sin embargo, difieren en el total de créditos: el programa de la Universidad de Antioquia requiere **180 créditos**, mientras que el de la Universidad Nacional de Colombia requiere **165 créditos**, es decir, 15 créditos menos que el de Antioquia.
---
=== claude-haiku-4

## 5. Generacion completa (8 combinaciones, con checkpoint)

Cada combinacion (`pipeline_retrieve x modelo x prompt`) se guarda en su propio checkpoint apenas termina, para no perder trabajo si algo falla a mitad de camino. Si vuelves a correr, las combinaciones ya generadas se leen del checkpoint sin volver a llamar al LLM.


In [7]:
RUTA_CHECKPOINTS_LLM = Path("./validacion/checkpoints_llm")
RUTA_CHECKPOINTS_LLM.mkdir(parents=True, exist_ok=True)

RUTA_SALIDA_LLM = "./validacion/resultados_generacion_llm.json"

# None = usa las preguntas filtradas completas. Pon un numero chico (ej. 10)
# para una corrida de prueba rapida antes de lanzar todo.
TAMANO_MUESTRA_LLM = None


def generar_para_combinacion(nombre_pipeline, resultados_pipeline, nombre_modelo, openrouter_id, nombre_prompt, funcion_prompt):
    etiqueta = f"{nombre_pipeline}__{nombre_modelo}__{nombre_prompt}"
    ruta_checkpoint = RUTA_CHECKPOINTS_LLM / f"{etiqueta}.json"

    if ruta_checkpoint.exists():
        print(f"  [{etiqueta}] ya generado antes, se reutiliza el checkpoint.")
        with open(ruta_checkpoint, "r", encoding="utf-8") as f:
            return json.load(f)

    preguntas_a_usar = resultados_pipeline[:TAMANO_MUESTRA_LLM] if TAMANO_MUESTRA_LLM else resultados_pipeline

    salida = []
    for i, r in enumerate(preguntas_a_usar, 1):
        prompt = funcion_prompt(r["pregunta"], r["contextos_recuperados"])
        respuesta_cruda = llamar_llm(openrouter_id, prompt)
        salida.append({
            "id": r["id"],
            "pregunta": r["pregunta"],
            "respuesta_esperada": r.get("respuesta_esperada"),
            "evidencia_gold": r.get("evidencia_gold"),
            "tipo": r.get("tipo"),
            "dificultad": r.get("dificultad"),
            "tipo_evidencia": r.get("tipo_evidencia"),
            "contextos_usados": r["contextos_recuperados"],
            "respuesta_generada_cruda": respuesta_cruda,
            "respuesta_generada": extraer_respuesta_final(respuesta_cruda, nombre_prompt),
        })
        if i % 25 == 0 or i == len(preguntas_a_usar):
            print(f"    [{etiqueta}] {i}/{len(preguntas_a_usar)}")

    with open(ruta_checkpoint, "w", encoding="utf-8") as f:
        json.dump(salida, f, ensure_ascii=False, indent=2)
    print(f"  [{etiqueta}] listo, {len(salida)} respuestas guardadas en {ruta_checkpoint}")
    return salida

In [8]:
combinaciones_resultado = []

for nombre_pipeline, resultados_pipeline in PIPELINES_RETRIEVE.items():
    for nombre_modelo, info_modelo in MODELOS_LLM.items():
        for nombre_prompt, funcion_prompt in ESTILOS_PROMPT.items():
            print(f"\n=== {nombre_pipeline} | {nombre_modelo} | prompt={nombre_prompt} ===")
            resultados_generados = generar_para_combinacion(
                nombre_pipeline, resultados_pipeline, nombre_modelo,
                info_modelo["openrouter_id"], nombre_prompt, funcion_prompt,
            )
            combinaciones_resultado.append({
                "pipeline_retrieve": nombre_pipeline,
                "modelo_llm": nombre_modelo,
                "openrouter_id": info_modelo["openrouter_id"],
                "rol_modelo": info_modelo["rol"],
                "estilo_prompt": nombre_prompt,
                "total_preguntas": len(resultados_generados),
                "resultados": resultados_generados,
            })

salida_final = {
    "dataset_fuente": "dataset_validacion_final.json",
    "criterio_filtro": "preguntas con llm_context_precision_with_reference > 0 y context_recall > 0 en AMBOS pipelines de retrieve (openai cs800 denso, BGE-m3 hibrido cs800)",
    "total_preguntas_filtradas": len(resultados_openai),
    "combinaciones": combinaciones_resultado,
}

Path(RUTA_SALIDA_LLM).parent.mkdir(parents=True, exist_ok=True)
with open(RUTA_SALIDA_LLM, "w", encoding="utf-8") as f:
    json.dump(salida_final, f, ensure_ascii=False, indent=2)

print(f"\nListo. {len(combinaciones_resultado)} combinaciones guardadas en {RUTA_SALIDA_LLM}")


=== openai_cs800 | claude-sonnet-5 | prompt=directo ===
    [openai_cs800__claude-sonnet-5__directo] 25/196
    [openai_cs800__claude-sonnet-5__directo] 50/196
    [openai_cs800__claude-sonnet-5__directo] 75/196
    [openai_cs800__claude-sonnet-5__directo] 100/196
    [openai_cs800__claude-sonnet-5__directo] 125/196
    [openai_cs800__claude-sonnet-5__directo] 150/196
    [openai_cs800__claude-sonnet-5__directo] 175/196
    [openai_cs800__claude-sonnet-5__directo] 196/196
  [openai_cs800__claude-sonnet-5__directo] listo, 196 respuestas guardadas en validacion\checkpoints_llm\openai_cs800__claude-sonnet-5__directo.json

=== openai_cs800 | claude-sonnet-5 | prompt=estructurado ===
    [openai_cs800__claude-sonnet-5__estructurado] 25/196
    [openai_cs800__claude-sonnet-5__estructurado] 50/196
    [openai_cs800__claude-sonnet-5__estructurado] 75/196
    [openai_cs800__claude-sonnet-5__estructurado] 100/196
    [openai_cs800__claude-sonnet-5__estructurado] 125/196
    [openai_cs800__claud

In [9]:
# Sanity check rapido
tamano_mb = Path(RUTA_SALIDA_LLM).stat().st_size / (1024 * 1024)
print(f"Archivo: {RUTA_SALIDA_LLM} ({tamano_mb:.2f} MB)")
for c in combinaciones_resultado:
    n_con_respuesta = sum(1 for r in c["resultados"] if r["respuesta_generada"])
    print(f"  {c['pipeline_retrieve']:22s} | {c['modelo_llm']:16s} | {c['estilo_prompt']:12s} -> {n_con_respuesta}/{c['total_preguntas']} con respuesta")

Archivo: ./validacion/resultados_generacion_llm.json (24.41 MB)
  openai_cs800           | claude-sonnet-5  | directo      -> 196/196 con respuesta
  openai_cs800           | claude-sonnet-5  | estructurado -> 196/196 con respuesta
  openai_cs800           | claude-haiku-4.5 | directo      -> 196/196 con respuesta
  openai_cs800           | claude-haiku-4.5 | estructurado -> 196/196 con respuesta
  bge_m3_hibrido_cs800   | claude-sonnet-5  | directo      -> 196/196 con respuesta
  bge_m3_hibrido_cs800   | claude-sonnet-5  | estructurado -> 196/196 con respuesta
  bge_m3_hibrido_cs800   | claude-haiku-4.5 | directo      -> 196/196 con respuesta
  bge_m3_hibrido_cs800   | claude-haiku-4.5 | estructurado -> 196/196 con respuesta


## Notas finales

- `resultados_generacion_llm.json` queda organizado en `combinaciones` (8: 2 pipelines x 2 modelos x 2 prompts), cada una con `pregunta`, `respuesta_esperada`, `evidencia_gold`, `contextos_usados` y `respuesta_generada` — listo para evaluar con RAGAS (`faithfulness`, `answer_relevancy`, etc. en la siguiente etapa).
- Para el prompt `estructurado`, `respuesta_generada` ya viene recortada a solo el contenido despues de "Respuesta final:" (mas facil de evaluar); `respuesta_generada_cruda` guarda la salida completa del modelo, con los pasos intermedios, por si quieres inspeccionar el razonamiento.
- Si `TAMANO_MUESTRA_LLM` no es `None` cuando corriste esto, los checkpoints quedan marcados con esa muestra chica — borra la carpeta `checkpoints_llm` (o los archivos puntuales de esa combinacion) antes de correr con `TAMANO_MUESTRA_LLM = None` para la corrida completa, o vas a quedarte con los resultados de la muestra chica en el archivo final.
- `temperature=0` en todas las llamadas, para que la comparacion entre modelos/prompts no tenga ruido adicional de muestreo aleatorio.
